In [1]:
# ---------------------------------------------------------
# Cell 1: Imports & Setup
# ---------------------------------------------------------
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import the new micro-model
from models.weight_cnn import DASWeightCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/home/chao/miniconda3/envs/au_sable/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Using device: cuda


In [2]:
# ---------------------------------------------------------
# Cell 2: Data Loading
# ---------------------------------------------------------
df = pd.read_csv('data/weight_labels.csv')

# Ensure paths point to denoised arrays
df['data_path'] = df['sample_id'].apply(
    lambda x: f"data/denoised/denoised_sample_{str(x).zfill(6)}.npy"
)

print(f"Total weight samples available: {len(df)}")
display(df[['sample_id', 'vehicle_type', 'weight', 'model']].head(10))

Total weight samples available: 70


,sample_id,vehicle_type,weight,model
0,4,suv,5325,Chevrolet Silverado 1500
1,5,suv,5325,Chevrolet Silverado 1500
2,6,suv,5325,Chevrolet Silverado 1500
3,7,suv,5325,Chevrolet Silverado 1500
4,8,background,0,NaN
5,9,suv,4050,Tesla Model 3
6,10,suv,4050,Tesla Model 3
7,11,suv,4050,Tesla Model 3
8,20,suv,3425,Honda CR-V
9,21,suv,3425,Honda CR-V


In [3]:
# ---------------------------------------------------------
# Cell 3: PyTorch Dataset & Split
# ---------------------------------------------------------
class DASWeightDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x_np = np.load(row['data_path']).astype(np.float32)
        x_tensor = torch.from_numpy(x_np).unsqueeze(0)
        
        # NO MORE LOGS. Just straight division.
        scaled_weight = row['weight'] / 1000.0
        y_tensor = torch.tensor([scaled_weight], dtype=torch.float32)
        
        return x_tensor, y_tensor

dataset = DASWeightDataset(df)

# Because we only have 27 samples, a standard split leaves tiny validation sets.
# Let's do roughly 19 Train / 4 Val / 4 Test
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset, [train_size, val_size, test_size], 
    generator=torch.Generator().manual_seed(42)
)

# Tiny batch size since we have tiny data
batch_size = 4 
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print(f"Split sizes -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Split sizes -> Train: 49 | Val: 10 | Test: 11


In [4]:
# ---------------------------------------------------------
# Cell 4: Model Initialization & Regularized Optimizer
# ---------------------------------------------------------
sample_x, _ = train_ds[0]
num_spatial_channels = sample_x.shape[1] 

# Initialize the lightweight CNN
model = DASWeightCNN(spatial_channels=num_spatial_channels).to(device)

# --- In Cell 4, update the criterion and optimizer ---
# L1Loss (Mean Absolute Error) is linear and perfectly stable for extreme outliers
criterion = nn.L1Loss() 

# REMOVED weight_decay=1e-4 so it stops forcing predictions to zero
optimizer = optim.Adam(model.parameters(), lr=7e-4)

In [ ]:
# Load the trained model saved by: python train.py --task weight
model.load_state_dict(torch.load("results/weight/best_model.pt", map_location=device))
model.eval()
print("Model loaded from results/weight/best_model.pt")

In [6]:
# ---------------------------------------------------------
# Cell 6: Evaluation (Restoring Actual Pounds)
# ---------------------------------------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error

model.eval()
actuals, predictions = [], []

# --- In Cell 6, update the evaluation loop ---
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = model(x)
        
# --- In Cell 6, update the evaluation un-scaling ---
        # Reverse the division!
        actual_lbs = y.numpy().flatten() * 1000.0
        raw_pred_lbs = preds.cpu().numpy().flatten() * 1000.0
        
        # Snap negatives to 0
        pred_lbs = np.clip(raw_pred_lbs, a_min=0, a_max=None)
        
        actuals.extend(actual_lbs.tolist())
        predictions.extend(pred_lbs.tolist())

# ... (The rest of Cell 6 remains the same)


actuals_arr = np.array(actuals)
preds_arr = np.array(predictions)

print("--- Test Set Predictions (in lbs) ---")
for act, pred in zip(actuals, predictions):
    print(f"Actual: {act:7.0f} lbs | Predicted: {pred:7.0f} lbs | Error: {abs(act-pred):5.0f} lbs")

mae = mean_absolute_error(actuals_arr, preds_arr)
rmse = np.sqrt(mean_squared_error(actuals_arr, preds_arr))

print("\n--- Final Weight Metrics ---")
print(f"Mean Absolute Error:   {mae:.0f} lbs")
print(f"Root Mean Sq. Error:   {rmse:.0f} lbs")

--- Test Set Predictions (in lbs) ---
Actual:       0 lbs | Predicted:       4 lbs | Error:     4 lbs
Actual:    5325 lbs | Predicted:    3460 lbs | Error:  1865 lbs
Actual:    5150 lbs | Predicted:    4369 lbs | Error:   781 lbs
Actual:    3425 lbs | Predicted:    2993 lbs | Error:   432 lbs
Actual:       0 lbs | Predicted:       6 lbs | Error:     6 lbs
Actual:    4515 lbs | Predicted:    3614 lbs | Error:   901 lbs
Actual:       0 lbs | Predicted:       5 lbs | Error:     5 lbs
Actual:       0 lbs | Predicted:      13 lbs | Error:    13 lbs
Actual:       0 lbs | Predicted:       6 lbs | Error:     6 lbs
Actual:       0 lbs | Predicted:       6 lbs | Error:     6 lbs
Actual:   29000 lbs | Predicted:   25475 lbs | Error:  3525 lbs

--- Final Weight Metrics ---
Mean Absolute Error:   686 lbs
Root Mean Sq. Error:   1262 lbs


In [ ]:
import matplotlib.pyplot as plt
from Utilities import plot_das_data
from config import DAS_FILE
from DAS import DAS

das = DAS(DAS_FILE)
dx = das.meta['dx']
dt = das.meta['dt']

sample_idx = 2
x_tensor, y_tensor = test_ds[sample_idx]

model.eval()
with torch.no_grad():
    x_input = x_tensor.unsqueeze(0).to(device)
    raw_pred = model(x_input)

actual_weight = y_tensor.item() * 1000.0
pred_weight = max(0, raw_pred.item() * 1000.0)

plot_data = x_tensor.squeeze().numpy()
channels = np.arange(plot_data.shape[0])

fig, ax = plt.subplots(figsize=(12, 6))
plot_title = f"Test Sample | Actual: {actual_weight:,.0f} lbs | Predicted: {pred_weight:,.0f} lbs"
plot_das_data(data=plot_data, channels=channels, dx=dx, dt=dt,
              title=plot_title, ax=ax, fig=fig, show=False)
plt.tight_layout()
plt.show()